# Voice-specific causal feature search (Qwen2.5-0.5B)

This notebook separates **generic grammatical voice** from the mapping learned by the voice LoRA.

1. Run 100 matched active/passive pairs through the unadapted base and voice adapter.
2. At every layer estimate
   `[(active − passive)voice − (active − passive)base]`.
3. Project that direction out on held-out pairs and identify the layer that moves the answer-logit gap closest to base.
4. At the winning layer, use the existing layer-18 SAE or train a new base-activation SAE.
5. Rank SAE features using voice-specific paired difference, sign consistency, activation prevalence, and decoder-scaled magnitude.
6. Ablate the top 20 individually at the final-answer position with CoTs fixed.
7. Compare cumulative top-1/3/6/10 feature ablations, a negative control, and exact residual-direction projection.

Pairs 1–50 estimate directions and rank features; pairs 51–100 measure interventions. This is an exploratory screen—winning features still require a fresh confirmatory dataset.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"
!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate tqdm matplotlib
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer
import hashlib, json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
PAIRS_PATH = Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl")
EXISTING_SAE = Path("sparse_autoencoders/artifacts/ethics_l18/sae.pt")
OUT = Path("sparse_autoencoders/artifacts/voice_feature_search")
OUT.mkdir(parents=True, exist_ok=True)
assert PAIRS_PATH.is_file() and EXISTING_SAE.is_file()

print("Upload paired-voice adapter_config.json and adapter_model.safetensors")
uploaded = files.upload()
VOICE_ADAPTER.mkdir(parents=True, exist_ok=True)
for name, data in uploaded.items():
    if Path(name).name in {"adapter_config.json", "adapter_model.safetensors", "training_log.json"}:
        (VOICE_ADAPTER / Path(name).name).write_bytes(data)
assert (VOICE_ADAPTER / "adapter_config.json").is_file()
assert (VOICE_ADAPTER / "adapter_model.safetensors").is_file()
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(VOICE_ADAPTER)
VOICE_SHA256 = hashlib.sha256((VOICE_ADAPTER / "adapter_model.safetensors").read_bytes()).hexdigest()
print("Voice adapter SHA256:", VOICE_SHA256)

In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers, train_sae
from sparse_autoencoders.sae import SparseAutoencoder
from evaluation.evaluate_ethics_morality import build_prompt

DEVICE = torch.device("cuda")
BATCH_SIZE = 16
rows = [json.loads(line) for line in PAIRS_PATH.read_text().splitlines() if line.strip()]
groups = {}
for row in rows:
    groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
assert len(groups) == 100 and all(set(pair) == {"active", "passive"} for pair in groups.values())

pairs = []
for pair_index, pair in sorted(groups.items()):
    active, passive = pair["active"], pair["passive"]
    assert active["scenario"] == passive["scenario"]
    assert active["sentence_stances"] == passive["sentence_stances"]
    pairs.append((pair_index, active, passive))

def readout_text(row):
    return f"{build_prompt(row)} {row['chain_of_thought']}\nFinal answer:"

active_texts = [readout_text(active) for _, active, _ in pairs]
passive_texts = [readout_text(passive) for _, _, passive in pairs]
DISCOVERY = torch.arange(0, 50)
VALIDATION = torch.arange(50, 100)
print("50 discovery pairs + 50 intervention-validation pairs")

In [ ]:
def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids

@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(layer_rows) for layer_rows in cached], dim=1)
    return activations, torch.cat(margins)

@torch.no_grad()
def evaluate_margins(model, tokenizer, texts, *, layer, direction=None, center=None, sae=None, feature=None):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE) if direction is not None else None
    center = center.to(DEVICE) if center is not None else None
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        if direction is not None:
            coefficient = ((target - center) * direction).sum(-1, keepdim=True)
            patched_target = target - coefficient * direction
        else:
            z = sae.encode(target)[:, feature]
            patched_target = target - z[:, None] * sae.decoder.weight[:, feature][None, :]
        patched = hidden.clone()
        patched[index, positions] = patched_target.to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)

In [ ]:
print("Collecting unadapted-base activations...")
base_tokenizer, base_model = load_model(BASE_MODEL, DEVICE)
base_active_h, base_active_margin = collect_all_layers(base_model, base_tokenizer, active_texts)
base_passive_h, base_passive_margin = collect_all_layers(base_model, base_tokenizer, passive_texts)
del base_model, base_tokenizer
gc.collect(); torch.cuda.empty_cache()

print("Collecting voice-adapter activations...")
voice_tokenizer, voice_model = load_model(str(VOICE_ADAPTER), DEVICE)
voice_active_h, voice_active_margin = collect_all_layers(voice_model, voice_tokenizer, active_texts)
voice_passive_h, voice_passive_margin = collect_all_layers(voice_model, voice_tokenizer, passive_texts)
assert voice_active_h.shape == base_active_h.shape == (100, 24, 896)

torch.save({
    "pair_indices": [pair_index for pair_index, _, _ in pairs],
    "base_active_h": base_active_h, "base_passive_h": base_passive_h,
    "voice_active_h": voice_active_h, "voice_passive_h": voice_passive_h,
    "base_active_margin": base_active_margin, "base_passive_margin": base_passive_margin,
    "voice_active_margin": voice_active_margin, "voice_passive_margin": voice_passive_margin,
}, OUT / "all_layer_activations.pt")
print("Saved", OUT / "all_layer_activations.pt")

In [ ]:
# Estimate each direction on discovery pairs, then causally project it out on validation pairs.
def gap(active_margin, passive_margin, indices):
    return float(active_margin[indices].mean() - passive_margin[indices].mean())

base_gap = gap(base_active_margin, base_passive_margin, VALIDATION)
voice_gap = gap(voice_active_margin, voice_passive_margin, VALIDATION)
voice_distance = abs(voice_gap - base_gap)
layer_results = []
validation_active_texts = active_texts[50:]
validation_passive_texts = passive_texts[50:]

for layer in range(24):
    paired_dod = (
        (voice_active_h[DISCOVERY, layer] - voice_passive_h[DISCOVERY, layer])
        - (base_active_h[DISCOVERY, layer] - base_passive_h[DISCOVERY, layer])
    )
    direction = F.normalize(paired_dod.mean(0), dim=0)
    center = torch.cat([
        voice_active_h[DISCOVERY, layer], voice_passive_h[DISCOVERY, layer]
    ]).mean(0)
    projected_active = evaluate_margins(
        voice_model, voice_tokenizer, validation_active_texts,
        layer=layer, direction=direction, center=center,
    )
    projected_passive = evaluate_margins(
        voice_model, voice_tokenizer, validation_passive_texts,
        layer=layer, direction=direction, center=center,
    )
    projected_gap = float(projected_active.mean() - projected_passive.mean())
    recovery = 1.0 - abs(projected_gap - base_gap) / max(voice_distance, 1e-8)
    rule_accuracy = float(torch.cat([projected_active > 0, projected_passive < 0]).float().mean())
    layer_results.append({
        "layer": layer,
        "base_gap": base_gap,
        "voice_gap": voice_gap,
        "projected_gap": projected_gap,
        "recovery_toward_base": recovery,
        "fixed_voice_rule_accuracy": rule_accuracy,
        "direction_norm_before_normalization": float(paired_dod.mean(0).norm()),
    })
    print(f"L{layer:02d} projected_gap={projected_gap:8.3f} recovery={recovery:7.3f} rule_acc={rule_accuracy:.3f}")

BEST_LAYER = max(layer_results, key=lambda row: row["recovery_toward_base"])["layer"]
(OUT / "layer_scan.json").write_text(json.dumps(layer_results, indent=2))
print(f"Best causal-recovery layer: {BEST_LAYER}")

In [ ]:
# Use the existing layer-18 SAE when possible; otherwise train a layer-specific SAE on base-model token activations.
del voice_model
gc.collect(); torch.cuda.empty_cache()
SAE_DIR = OUT / f"sae_l{BEST_LAYER}"
SAE_DIR.mkdir(exist_ok=True)

@torch.no_grad()
def collect_token_activations(model, tokenizer, texts, layer):
    mask = None
    chunks = []
    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        chunks.append(hidden[mask.bool()].detach().float().cpu())
    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(DEVICE)
            mask = inputs["attention_mask"]
            model(**inputs, use_cache=False)
    finally:
        handle.remove()
    return torch.cat(chunks)

if BEST_LAYER == 18:
    checkpoint = torch.load(EXISTING_SAE, map_location="cpu", weights_only=True)
    sae = SparseAutoencoder(896, int(checkpoint["dict_size"]))
    sae.load_state_dict(checkpoint["state_dict"])
    sae.to(DEVICE).eval()
    SAE_SOURCE = str(EXISTING_SAE)
else:
    base_tokenizer, base_model = load_model(BASE_MODEL, DEVICE)
    discovery_texts = active_texts[:50] + passive_texts[:50]
    token_acts = collect_token_activations(base_model, base_tokenizer, discovery_texts, BEST_LAYER)
    del base_model, base_tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print("Training SAE on", token_acts.shape[0], "discovery-split base-model token activations")
    sae = train_sae(token_acts, 7168, 3000, 256, 1e-3, 1e-3, DEVICE)
    torch.save({"layer": BEST_LAYER, "dict_size": 7168, "state_dict": sae.state_dict()}, SAE_DIR / "sae.pt")
    SAE_SOURCE = str(SAE_DIR / "sae.pt")
    del token_acts

print("SAE source:", SAE_SOURCE)

In [ ]:
# Rank features on discovery pairs only.
@torch.no_grad()
def encode_acts(activations, batch_size=64):
    return torch.cat([
        sae.encode(activations[start:start + batch_size].to(DEVICE).float()).cpu()
        for start in range(0, len(activations), batch_size)
    ])

base_active_z = encode_acts(base_active_h[:, BEST_LAYER])
base_passive_z = encode_acts(base_passive_h[:, BEST_LAYER])
voice_active_z = encode_acts(voice_active_h[:, BEST_LAYER])
voice_passive_z = encode_acts(voice_passive_h[:, BEST_LAYER])
paired_feature_dod = (
    (voice_active_z[DISCOVERY] - voice_passive_z[DISCOVERY])
    - (base_active_z[DISCOVERY] - base_passive_z[DISCOVERY])
)
mean_difference = paired_feature_dod.mean(0)
sign = mean_difference.sign()
consistency = ((paired_feature_dod * sign) > 0).float().mean(0)
prevalence = torch.cat([voice_active_z[DISCOVERY], voice_passive_z[DISCOVERY]]).gt(1e-6).float().mean(0)
decoder_norm = sae.decoder.weight.detach().float().cpu().norm(dim=0)
scaled_magnitude = mean_difference.abs() * decoder_norm
eligible = (consistency >= 0.60) & (prevalence >= 0.05)
eligible_count = int(eligible.sum())
assert eligible_count >= 20, "Too few stable active features; relax thresholds explicitly"
ranking_score = scaled_magnitude.masked_fill(~eligible, -torch.inf)
values, indices = torch.topk(ranking_score, min(200, eligible_count))

feature_ranking = []
for rank, (score, feature) in enumerate(zip(values, indices), 1):
    feature_ranking.append({
        "rank": rank,
        "feature": int(feature),
        "voice_specific_mean_difference": float(mean_difference[feature]),
        "sign_consistency": float(consistency[feature]),
        "activation_prevalence": float(prevalence[feature]),
        "decoder_norm": float(decoder_norm[feature]),
        "decoder_scaled_magnitude": float(score),
    })
TOP20 = [row["feature"] for row in feature_ranking[:20]]
(OUT / "feature_ranking.json").write_text(json.dumps(feature_ranking, indent=2))
print(f"{'rank':>4} {'feature':>8} {'Δvoice':>10} {'consistent':>11} {'prevalent':>10} {'scaled':>10}")
for row in feature_ranking[:20]:
    print(f"{row['rank']:4d} {row['feature']:8d} {row['voice_specific_mean_difference']:10.3f} {row['sign_consistency']:11.2f} {row['activation_prevalence']:10.2f} {row['decoder_scaled_magnitude']:10.3f}")

In [ ]:
# Causally screen each top feature on validation pairs with the CoTs fixed.
voice_tokenizer, voice_model = load_model(str(VOICE_ADAPTER), DEVICE)
base_validation = torch.cat([base_active_margin[VALIDATION], base_passive_margin[VALIDATION]])
voice_validation = torch.cat([voice_active_margin[VALIDATION], voice_passive_margin[VALIDATION]])
baseline_distance = float((voice_validation - base_validation).abs().mean())
feature_effects = []

for rank, feature in enumerate(TOP20, 1):
    active_margin = evaluate_margins(
        voice_model, voice_tokenizer, validation_active_texts,
        layer=BEST_LAYER, sae=sae, feature=feature,
    )
    passive_margin = evaluate_margins(
        voice_model, voice_tokenizer, validation_passive_texts,
        layer=BEST_LAYER, sae=sae, feature=feature,
    )
    ablated = torch.cat([active_margin, passive_margin])
    ablated_gap = float(active_margin.mean() - passive_margin.mean())
    distance = float((ablated - base_validation).abs().mean())
    effect = {
        "screen_rank": rank,
        "feature": feature,
        "voice_gap_before": voice_gap,
        "voice_gap_after": ablated_gap,
        "gap_reduction": voice_gap - ablated_gap,
        "mean_logit_distance_to_base_before": baseline_distance,
        "mean_logit_distance_to_base_after": distance,
        "logit_recovery_toward_base": 1.0 - distance / max(baseline_distance, 1e-8),
        "fixed_voice_rule_accuracy": float(torch.cat([active_margin > 0, passive_margin < 0]).float().mean()),
        "label_changes": int((ablated.gt(0) != voice_validation.gt(0)).sum()),
    }
    feature_effects.append(effect)
    print(f"f{feature:4d} gap={ablated_gap:8.3f} recovery={effect['logit_recovery_toward_base']:7.3f} rule_acc={effect['fixed_voice_rule_accuracy']:.3f} flips={effect['label_changes']}")

feature_effects.sort(key=lambda row: row["logit_recovery_toward_base"], reverse=True)
(OUT / "single_feature_ablation.json").write_text(json.dumps(feature_effects, indent=2))
print("\nBest causal candidates:", [row["feature"] for row in feature_effects[:6]])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(
    [row["layer"] for row in layer_results],
    [row["recovery_toward_base"] for row in layer_results],
    marker="o", linewidth=1.8,
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axvline(BEST_LAYER, color="tab:red", linestyle="--", label=f"best: layer {BEST_LAYER}")
axes[0].set(title="Voice-specific residual direction", xlabel="Transformer layer", ylabel="Logit-gap recovery toward base")
axes[0].legend(frameon=False)

shown = list(reversed(feature_effects))
colors = ["tab:blue" if row["logit_recovery_toward_base"] > 0 else "tab:orange" for row in shown]
axes[1].barh(
    [str(row["feature"]) for row in shown],
    [row["logit_recovery_toward_base"] for row in shown],
    color=colors,
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(title=f"Single-feature causal screen at layer {BEST_LAYER}", xlabel="Mean-logit recovery toward base", ylabel="SAE feature ID")
fig.suptitle("Voice-rule causal localization and SAE feature screening", fontsize=14)
fig.savefig(OUT / "voice_feature_search.png", dpi=300, bbox_inches="tight")
plt.show()

metadata = {
    "base_model": BASE_MODEL,
    "voice_adapter": str(VOICE_ADAPTER),
    "voice_adapter_sha256": VOICE_SHA256,
    "pairs": 100,
    "discovery_pairs": 50,
    "validation_pairs": 50,
    "best_layer": BEST_LAYER,
    "sae_source": SAE_SOURCE,
    "base_answer_logit_gap": base_gap,
    "voice_answer_logit_gap": voice_gap,
    "ranking_thresholds": {"sign_consistency": 0.60, "activation_prevalence": 0.05},
    "top20_screened": TOP20,
    "best_six_after_causal_screen": [row["feature"] for row in feature_effects[:6]],
}
(OUT / "experiment.json").write_text(json.dumps(metadata, indent=2))
print("Initial search artifacts saved; grouped confirmation follows below.")

## Grouped fixed-CoT confirmation

Run this section after the single-feature screen. It compares:

- Exact residual-direction projection at the winning layer (primary)
- Cumulative top 1, 3, 6, and 10 positively recovering SAE features
- Six negatively recovering SAE features as a control

The combinations are derived dynamically because retraining an SAE can change feature IDs. These 50 validation pairs were used for single-feature screening, so grouped results remain exploratory rather than independent confirmation.

In [ ]:
@torch.no_grad()
def evaluate_feature_group(model, tokenizer, texts, *, layer, sae, features):
    tokenizer.padding_side = "right"
    positions = None
    feature_ids = torch.tensor(features, device=DEVICE, dtype=torch.long)
    decoder = sae.decoder.weight.index_select(1, feature_ids)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        selected = sae.encode(target).index_select(1, feature_ids)
        patched = hidden.clone()
        patched[index, positions] = (target - selected @ decoder.T).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)

positive_features = [
    row["feature"] for row in feature_effects
    if row["logit_recovery_toward_base"] > 0
]
assert len(positive_features) >= 10, "Fewer than ten positive features; inspect the screen before grouping"
FEATURE_GROUPS = {
    "causal_top1": positive_features[:1],
    "causal_top3": positive_features[:3],
    "causal_top6": positive_features[:6],
    "causal_top10": positive_features[:10],
    "anti_recovery_control6": [row["feature"] for row in feature_effects[-6:]],
}
print("Grouped arms:", FEATURE_GROUPS)

def intervention_summary(name, active_margin, passive_margin, features=None):
    combined = torch.cat([active_margin, passive_margin])
    distance = float((combined - base_validation).abs().mean())
    gap_after = float(active_margin.mean() - passive_margin.mean())
    return {
        "arm": name,
        "features": features,
        "voice_gap_before": voice_gap,
        "voice_gap_after": gap_after,
        "gap_reduction": voice_gap - gap_after,
        "gap_recovery_toward_base": 1.0 - abs(gap_after - base_gap) / max(voice_distance, 1e-8),
        "mean_logit_distance_to_base_before": baseline_distance,
        "mean_logit_distance_to_base_after": distance,
        "logit_recovery_toward_base": 1.0 - distance / max(baseline_distance, 1e-8),
        "fixed_voice_rule_accuracy": float(torch.cat([active_margin > 0, passive_margin < 0]).float().mean()),
        "label_changes": int((combined.gt(0) != voice_validation.gt(0)).sum()),
    }

group_results = [intervention_summary(
    "unablated_voice", voice_active_margin[VALIDATION], voice_passive_margin[VALIDATION]
)]
paired_dod = (
    (voice_active_h[DISCOVERY, BEST_LAYER] - voice_passive_h[DISCOVERY, BEST_LAYER])
    - (base_active_h[DISCOVERY, BEST_LAYER] - base_passive_h[DISCOVERY, BEST_LAYER])
)
residual_direction = F.normalize(paired_dod.mean(0), dim=0)
residual_center = torch.cat([
    voice_active_h[DISCOVERY, BEST_LAYER], voice_passive_h[DISCOVERY, BEST_LAYER]
]).mean(0)
projected_active = evaluate_margins(
    voice_model, voice_tokenizer, validation_active_texts,
    layer=BEST_LAYER, direction=residual_direction, center=residual_center,
)
projected_passive = evaluate_margins(
    voice_model, voice_tokenizer, validation_passive_texts,
    layer=BEST_LAYER, direction=residual_direction, center=residual_center,
)
group_results.append(intervention_summary(
    "exact_residual_projection", projected_active, projected_passive
))

for name, features in FEATURE_GROUPS.items():
    active_margin = evaluate_feature_group(
        voice_model, voice_tokenizer, validation_active_texts,
        layer=BEST_LAYER, sae=sae, features=features,
    )
    passive_margin = evaluate_feature_group(
        voice_model, voice_tokenizer, validation_passive_texts,
        layer=BEST_LAYER, sae=sae, features=features,
    )
    group_results.append(intervention_summary(name, active_margin, passive_margin, features))

(OUT / "group_ablation.json").write_text(json.dumps(group_results, indent=2))
for row in group_results:
    print(
        f"{row['arm']:26s} gap={row['voice_gap_after']:8.3f} "
        f"gap_recovery={row['gap_recovery_toward_base']:7.3f} "
        f"sample_recovery={row['logit_recovery_toward_base']:7.3f} "
        f"rule_acc={row['fixed_voice_rule_accuracy']:.3f} flips={row['label_changes']}"
    )

In [ ]:
import shutil
import numpy as np

labels = [row["arm"].replace("_", "\n") for row in group_results]
x = np.arange(len(group_results))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

axes[0].bar(x, [row["voice_gap_after"] for row in group_results], color="tab:blue")
axes[0].axhline(base_gap, color="black", linestyle="--", linewidth=1.5, label="Unadapted base")
axes[0].axhline(voice_gap, color="tab:red", linestyle=":", linewidth=1.5, label="Voice adapter")
axes[0].set(title="Answer-logit gap after intervention", ylabel="mean logit(1) − logit(0) gap", xticks=x, xticklabels=labels)
axes[0].legend(frameon=False)

width = 0.38
axes[1].bar(
    x - width / 2, [row["gap_recovery_toward_base"] for row in group_results],
    width, label="Aggregate-gap recovery", color="tab:green",
)
axes[1].bar(
    x + width / 2, [row["logit_recovery_toward_base"] for row in group_results],
    width, label="Per-example recovery", color="tab:purple",
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Recovery toward unadapted behavior", ylabel="fraction recovered", xticks=x, xticklabels=labels)
axes[1].legend(frameon=False)
for axis in axes:
    axis.tick_params(axis="x", labelrotation=25, labelsize=8)

fig.suptitle(f"Fixed-CoT voice intervention at layer {BEST_LAYER}", fontsize=14)
fig.savefig(OUT / "group_ablation.png", dpi=300, bbox_inches="tight")
plt.show()

metadata["grouped_fixed_cot"] = {
    "feature_groups": FEATURE_GROUPS,
    "primary_arm": "exact_residual_projection",
    "selection_note": "SAE groups were selected using the same validation half and are exploratory.",
}
(OUT / "experiment.json").write_text(json.dumps(metadata, indent=2))
archive = shutil.make_archive("/content/voice_feature_search_05b", "zip", root_dir=OUT)
files.download(archive)
print("Downloaded", archive)